# triangle-barycentric composite — cx19: barycentric ray-triangle intersection via batched linalg.solve

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `triangle-barycentric`, `linalg-solve-batched`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "triangle-barycentric"
DD_ATOM_IDS = ["triangle-barycentric", "linalg-solve-batched"]
DD_SUBTOPICS = ["Geometry: Barycentric coords", "PyTorch: Batched linalg.solve"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's ray-triangle intersection rewrites the geometric question 'does this ray hit this triangle?' as a 3x3 LINEAR SYSTEM in the unknowns `(s, u, v)`:

  `O + s D = A + u (B - A) + v (C - A)`

Rearranged: `[-D | (B-A) | (C-A)] [s; u; v] = O - A`. To run it over a batch of (ray, triangle) pairs you stack the LHS into a `(N, 3, 3)` tensor and the RHS into `(N, 3)`, then call `torch.linalg.solve(M, y)` once — vectorized, no Python loop.

The composition makes both atoms load-bearing: the barycentric formulation produces the system, and the batched solve is what makes it tractable over millions of pairs.

### Composite Exercise — barycentric ray-triangle intersection via batched linalg.solve

**Atoms exercised together**: `triangle-barycentric`, `linalg-solve-batched`

Implement `cx19_intersect_batched(rays, triangle)` that, for a batch of `N` rays and a single triangle, returns the `(N, 3)` tensor of `(s, u, v)` solutions to the barycentric system.

Inputs:
- `rays`: shape `(N, 2, 3)` — `rays[i, 0]` is origin `O_i`, `rays[i, 1]` is direction `D_i`.
- `triangle`: shape `(3, 3)` — rows are vertices `A`, `B`, `C`.

1. **Barycentric setup** — build the 3x3 system per ray:
   - Columns of `M_i`: `[-D_i, B-A, C-A]`.
   - RHS `y_i = O_i - A`.
2. **Batched solve** — call `t.linalg.solve(M, y)` on the stacked `(N, 3, 3)` and `(N, 3)` tensors. Return the `(N, 3)` `(s, u, v)` matrix.

The test cross-checks each row against a per-ray `t.linalg.solve` call (no batching) and verifies that plugging `(s, u, v)` back into the parametric equations recovers the same 3D point on both sides.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx19_intersect_batched(rays, triangle):
    raise NotImplementedError

def _test_cx19():
    # Case A: small N, hand-built rays + triangle.
    A = t.tensor([0.0, 0.0, 0.0])
    B = t.tensor([1.0, 0.0, 0.0])
    C = t.tensor([0.0, 1.0, 0.0])
    tri = t.stack([A, B, C], dim=0)  # (3, 3)
    # Two rays from z=1 shooting toward -z.
    O = t.tensor([[0.25, 0.25, 1.0], [2.0, 2.0, 1.0]])
    D = t.tensor([[0.0, 0.0, -1.0], [0.0, 0.0, -1.0]])
    rays = t.stack([O, D], dim=1)  # (2, 2, 3)
    out = cx19_intersect_batched(rays, tri)
    assert tuple(out.shape) == (2, 3), f'expected (2,3), got {tuple(out.shape)}'
    # Ray 0 hits at (0.25, 0.25, 0) => s=1, u=0.25, v=0.25.
    assert t.allclose(out[0], t.tensor([1.0, 0.25, 0.25]), atol=1e-5), f'ray0 sol: {out[0]}'
    # Ray 1 also lands on the plane at s=1 but outside the triangle (u=2, v=2).
    assert t.allclose(out[1], t.tensor([1.0, 2.0, 2.0]), atol=1e-5), f'ray1 sol: {out[1]}'

    # Case B: cross-check against per-ray solve on a random batch.
    t.manual_seed(7)
    N = 32
    rays_b = t.randn(N, 2, 3)
    tri_b = t.randn(3, 3)
    out_b = cx19_intersect_batched(rays_b, tri_b)
    assert tuple(out_b.shape) == (N, 3)
    A_b = tri_b[0]
    for i in range(N):
        O_i, D_i = rays_b[i, 0], rays_b[i, 1]
        M_i = t.stack([-D_i, tri_b[1] - A_b, tri_b[2] - A_b], dim=1)
        y_i = O_i - A_b
        ref = t.linalg.solve(M_i, y_i)
        assert t.allclose(out_b[i], ref, atol=1e-4), f'row {i} mismatch: {out_b[i]} vs {ref}'

    # Case C: plug (s, u, v) back into both sides — must match in 3D.
    s, u, v = out_b[0, 0], out_b[0, 1], out_b[0, 2]
    lhs = rays_b[0, 0] + s * rays_b[0, 1]
    rhs = tri_b[0] + u * (tri_b[1] - tri_b[0]) + v * (tri_b[2] - tri_b[0])
    assert t.allclose(lhs, rhs, atol=1e-4), f'parametric check failed: {lhs} vs {rhs}'
    _dd_passed.add('cx19')

_test_cx19()

<details><summary>Show solution — cx19</summary>

```python
def cx19_intersect_batched(rays, triangle):
    A, B, C = triangle[0], triangle[1], triangle[2]
    O = rays[..., 0, :]  # (N, 3)
    D = rays[..., 1, :]  # (N, 3)
    N = O.shape[0]
    # Atom A (triangle-barycentric): build the per-ray 3x3 system.
    # Columns are [-D, B-A, C-A]; stack along last dim so M has shape (N, 3, 3).
    col0 = -D                                       # (N, 3)
    col1 = (B - A).unsqueeze(0).expand(N, -1)       # (N, 3)
    col2 = (C - A).unsqueeze(0).expand(N, -1)       # (N, 3)
    M = t.stack([col0, col1, col2], dim=-1)         # (N, 3, 3)
    y = O - A                                       # (N, 3)
    # Atom B (linalg-solve-batched): one vectorised solve over the leading batch dim.
    return t.linalg.solve(M, y)                     # (N, 3)
```

Batched `t.linalg.solve` is what makes ARENA's million-ray intersection tractable — the per-ray Python loop in the test is here only as ground truth. The barycentric setup is the load-bearing piece: each column of `M` matches one term of the parametric equation, in exactly the order the unknowns `(s, u, v)` are stacked.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["Geometry: Barycentric coords", "PyTorch: Batched linalg.solve"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()